In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)


In [2]:
# Cell 1: Imports and setup
import warnings
import os
import sys
import math
import time
import datetime

import numpy as np
import dill
import scipy.stats
from IPython.display import display, Markdown as md

# Suppress JupyterViz warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*JupyterViz: `use_state` found within a nested function.*")
warnings.filterwarnings("ignore", module="solara.*")
warnings.filterwarnings("ignore", category=UserWarning)


# Your custom modules
from resistance_model import *



In [3]:
from IPython.display import Markdown as md
import numpy as np
import dill
import os
import scipy.stats

select = 0
displayMode = ["singleRunVisualize", "batchRun", "loadSession", "loadSensAnalysis"][select]

override_mesa_shapes() #built-in mesa draw doesn't have shapes - this overwrites with custom shape-including one


In [4]:
def agent_portrayal(agent):
    """Return color, size, and marker style for a given agent type."""

    # defaults
    color = "gray"
    marker = "."
    size = 30
    # ---------------- CITIZENS ----------------
    if agent.type == ATypes.CITIZEN:
        marker = "o"
        size = 30

        if agent.jailTerm:
            color = "white"                 # REMOVED from system
        elif agent.inProtest:
            color = "cyan"                  # ACTION
        elif agent.activeNV:
            color = "mediumaquamarine"              # ORGANIZED / MOBILIZED
        elif agent.grievanceLevel == "high":
            color = "lightblue"      # LATENT PRESSURE
        else:
            color = "lightcyan"             # BACKGROUND

    # ---------------- ACTIVISTS ----------------
    elif agent.type == ATypes.ACTIVIST:
        if agent.inProtest:
            color = "red"
        elif agent.jailTerm:
            color = "lightpink"
        else:
            color = "lightpink"

        marker = "." if agent.jailTerm else "o"

    # ---------------- POLICE ----------------
    elif agent.type == ATypes.POLICE:
        # state: defected vs loyal
        color = "lightgray" if agent.defect else "black"

        # trait: defection propensity
        if agent.defectLevel == "high":
            size = 30
            # size = 100
        else:
            size = 30

        marker = "v"

    # ---------------- PILLARS ----------------
    elif agent.type == ATypes.PILLAR:
        color = "thistle" if agent.defect else "fuchsia"

        if agent.defectLevel == "high":
            size = 30
        else:
            size = 30
        marker = "*"

    return {"color": color, "size": size, "marker": marker}


In [5]:
print("citizens", 1600*0.7)
print("police", 1600*0.04)
print("activists", 1600*0.008)
print("pillar", 1600*0.0153)


citizens 1120.0
police 64.0
activists 12.8
pillar 24.48


In [6]:
1133+84 

1217

In [7]:
64+24

88

In [8]:
import glob, dill

save_dir = 'NVResistance_LLMResults/'
# 2 conflicting paramters PctPillar (1.53 vs 0.85), ProtestCycle (8.9 vs 7)
model_params = {   # values from S1 table, chenoweth initial optimized values
    'MaxSteps':200, 'PctCitizen':70, 'PctActivist':0.8, 'PctPolice':4, 'PctPillar':1.53, 'MaxJailTerm':10,
    'StartLegitimacy':0.56, 'PctFindNV':40, 'PctTargetNV':25, 'PctKillNV':10, 'BackfireCoeff':0.99,
    'valueF':0.0706, 'ProtestCycle':8.9, 'ProtestDuration':1, 'nNVthreshold':1, 'DefectThresh':0.05,
    'PeerPressureNum':3.46, 'DelayStartMax':5, 'NVSuccessPct':10, 'PillarProxStrat':3,
    'DefectThreshStDev':0.01, 'ActivistSearchVision':10, 'PctFickle':100, 'PctImmedProtest':5,
    'dataCollectorType':displayMode
}


# overrides model_params, put sliders here
param_visual_overrides = {
    "NVSuccessPct":{ "type": "SliderFloat", "value": model_params['NVSuccessPct'], "label": "NV Win % (Pillars)", "min": 5, "max": 50, "step": 1}
}


if displayMode == "singleRunVisualize":
    for key in param_visual_overrides.keys():
        model_params[key] = param_visual_overrides[key]
    page = JupyterViz(
        NVResistanceModel,
        model_params,
        measures = [
            {
             "protest":"cyan",
             "activists":"red",
             "police":"black", 
             "pillars":"fuchsia", 
            #  "activeNV":"mediumaquamarine"
             },
            make_text(lambda x: f"### Model Seed: {x._seed}"),
        ],
        name = "",
        agent_portrayal = agent_portrayal,
        play_interval=0,
    )
elif displayMode == "batchRun":
    
    total_iterations = 10
    chunk_size = 2
    save_folder = "NVResistance_BatchResult_actvt"

    os.makedirs(save_folder, exist_ok=True)

    for start in range(0, total_iterations, chunk_size):
        end = start + chunk_size

        print(f"\n=== Running iterations {start} to {end} ===")

        results = batchrunner.batch_run(
            NVResistanceModel,
            model_params,
            number_processes=None,                     # keep None for multiprocessing
            iterations=chunk_size,             # number of runs in this chunk
            max_steps=model_params["MaxSteps"],
            display_progress=True
        )

        filename = f"multi_run_data_{start}-{end}.db"
        save_path = os.path.join(save_folder, filename)

        with open(save_path, "wb") as f:
            dill.dump(results, f)

        print(f"Saved → {save_path}")

elif displayMode == "loadSession":
    dill.load_session('multi_run_data.db')
    page = "Loaded Runs"

page



Cannot show ipywidgets in text

In [ ]:
select = 1   # batchRun
displayMode = ["singleRunVisualize", "batchRun", "loadSession"][select]

orchestrator = RunOrchestrator(
    base_config  = LogConfig(
        provider    = "huggingface_llama",
        prompt_type = "ZeroShot",
        memory      = "stateful",
        vision      = "vision-4",
        base_dir    = "SimulationLogs",
    ),
    params       = model_params,
    max_workers  = 1,    # must be 1 — HuggingFace loads model in main process
    start_run_id = 0,
)

summary = orchestrator.run(n_runs=3)   # start with 3 to verify it works
# ```

# ---

# ## Step 6 — First run will download the model

# The first time you run, HuggingFace downloads ~16GB of weights to your cache. This takes 10-20 minutes depending on your internet. Subsequent runs use the cached version and start in ~30 seconds.

# To control where the model is cached, add to your `.env`:
# ```
# HF_HOME=C:/Users/fmahnoor1/.cache/huggingface